Table 19 (DNA): Pooled Significance Test vs. LogReg Baseline

Table 4 reports single point-estimate Precision@10/Recall@10 values per drug/model (not averaged across CV folds, per its own caption), so a per-drug/per-fold Wilcoxon test (as used for Table 16/Task 1) is not possible here. Instead, following the same across-drug pooled approach used for Table 17 and the Task 2 lineage-delta significance test, we run one paired Wilcoxon signed-rank test per model (vs. LogReg), pairing each drug's Precision@10 (and, separately, Recall@10) across all 10 drugs. Holm-Bonferroni correction is applied within each metric's family of 4 models.

In [ ]:
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from pathlib import Path

RANDOM_SPLIT_DIR = Path("../results/test_output_summary/task2/random_split")
OUT_FILE = "../results/significance_testing/table19_lineage_split_task2_significance_testing.csv"

ARCH_FILES = {
    "SD-LogReg": "random_split_summmary_logreg.csv",
    "SD-CNN": "random_split_summary_sdcnn.csv",
    "Evo2-CNN-pca10": "random_split_summary_evo2_zeroshot_pca.csv",
    "Evo2-CNN-zero-shot": "random_split_summary_evo2_zeroshot_token.csv",
    "MD-CNN": "random_split_summary_mdcnn.csv",
}

BASELINE = "SD-LogReg"
MODELS = ["SD-CNN", "Evo2-CNN-pca10", "Evo2-CNN-zero-shot", "MD-CNN"]

In [6]:
def load_k10_data(arch_name, filename):
    """Load k=10 data from CSV file."""
    df = pd.read_csv(RANDOM_SPLIT_DIR / filename)
    k10 = df[df['k'] == 10][['drug', 'P@k', 'R@k']].copy()
    k10['drug'] = k10['drug'].str.upper()
    k10 = k10.set_index('drug')
    return k10

baseline_data = load_k10_data(BASELINE, ARCH_FILES[BASELINE])

rows = []
for model in MODELS:
    model_data = load_k10_data(model, ARCH_FILES[model])
    common_drugs = sorted(set(baseline_data.index) & set(model_data.index))
    
    baseline_p = baseline_data.loc[common_drugs, 'P@k'].values
    baseline_r = baseline_data.loc[common_drugs, 'R@k'].values
    model_p = model_data.loc[common_drugs, 'P@k'].values
    model_r = model_data.loc[common_drugs, 'R@k'].values
    
    _, p_precision = wilcoxon(model_p, baseline_p, alternative="two-sided")
    _, p_recall = wilcoxon(model_r, baseline_r, alternative="two-sided")
    
    rows.append({
        "model": model,
        "mean_baseline_precision": baseline_p.mean(),
        "mean_model_precision": model_p.mean(),
        "delta_precision": model_p.mean() - baseline_p.mean(),
        "p_precision": p_precision,
        "mean_baseline_recall": baseline_r.mean(),
        "mean_model_recall": model_r.mean(),
        "delta_recall": model_r.mean() - baseline_r.mean(),
        "p_recall": p_recall,
        "n_drugs": len(common_drugs),
    })

result = pd.DataFrame(rows).set_index("model")
result["p_precision_holm"] = multipletests(result["p_precision"], method="holm")[1]
result["p_recall_holm"] = multipletests(result["p_recall"], method="holm")[1]

result.reset_index().to_csv(OUT_FILE, index=False)
print("[OK] Completed pooled significance test (DNA Task 2 vs SD-LogReg baseline)")
print(f"[OK] Results saved to {OUT_FILE}")
result.round(4)

[OK] Completed pooled significance test (DNA Task 2 vs SD-LogReg baseline)
[OK] Results saved to ../results/significance_testing/table19_lineage_split_task2_significance_testing.csv


/home/saishradhamo_umass_edu/.local/lib/python3.8/site-packages/scipy/stats/_morestats.py:3414: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
/home/saishradhamo_umass_edu/.local/lib/python3.8/site-packages/scipy/stats/_morestats.py:3428: UserWarning: Sample size too small for normal approximation.
  warnings.warn("Sample size too small for normal approximation.")
/home/saishradhamo_umass_edu/.local/lib/python3.8/site-packages/scipy/stats/_morestats.py:3414: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
/home/saishradhamo_umass_edu/.local/lib/python3.8/site-packages/scipy/stats/_morestats.py:3428: UserWarning: Sample size too small for normal approximation.
  warnings.warn("Sample size too small for normal approximation.")


,mean_baseline_precision,mean_model_precision,delta_precision,p_precision,mean_baseline_recall,mean_model_recall,delta_recall,p_recall,n_drugs,p_precision_holm,p_recall_holm
model,,,,,,,,,,,
SD-CNN,0.3909,0.3909,0.0000,0.8576,0.2056,0.2505,0.0449,0.2600,11,0.8576,0.2600
Evo2-CNN-pca10,0.3909,0.2000,-0.1909,0.0076,0.2056,0.0952,-0.1103,0.0077,11,0.0227,0.0231
Evo2-CNN-zero-shot,0.3909,0.2273,-0.1636,0.0328,0.2056,0.1220,-0.0836,0.0858,11,0.0656,0.1717
MD-CNN,0.3909,0.1364,-0.2545,0.0049,0.2056,0.0763,-0.1293,0.0050,11,0.0196,0.0201
